Additional Features:
 - Excess style momentum
 - Excess style volatility (against each other)
 - correlation between factor returns (against benchmark)
 - Excess style skewness (against benchmark)
 - Relative momentum (factors relative to each other, trend scan - categorical features)
 


In [11]:
# run script from the data_input file
import os
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')

In [12]:
# Import libraries:
import numpy as np
import pandas as pd
from scipy.stats import kurtosis, skew


In [13]:
# Import data:
df_factors = pd.read_csv("factor_returns.csv", parse_dates=['Date'])
df_factors = df_factors.set_index('Date')

#df_vol = pd.read_csv("local_global_vol.csv", parse_dates=['Date'])
#df_vol = df_vol.set_index('Date')

#df = pd.concat([df_factors,df_vol], axis=1, join='inner')
df = df_factors.dropna()

print(df)
#print(df.dtypes)


            Momentum     Value   Quality
Date                                    
2007-01-22  0.000357  0.010732  0.014931
2007-01-29  0.021934  0.011995 -0.005972
2007-02-05 -0.014909 -0.007560  0.038414
2007-02-12  0.044639  0.039344  0.010788
2007-02-19  0.014797  0.010884 -0.045485
...              ...       ...       ...
2024-01-01 -0.020676 -0.025835 -0.022511
2024-01-08  0.002769 -0.000758  0.008803
2024-01-15 -0.014716 -0.035979  0.007137
2024-01-22  0.029946  0.027634 -0.013507
2024-01-29 -0.006574 -0.005330  0.014779

[885 rows x 3 columns]


In [14]:
import itertools as it

def factor_relative_core_features(df, window=12):

    feat = pd.DataFrame(index=df.index)
    factors = df.columns

    for a, b in it.combinations(factors, 2):
        name = f"{a.lower()}_vs_{b.lower()}"
        spread = df[a] - df[b]

        # cumulative relative performance
        cum_a = (1 + df[a]).cumprod()
        cum_b = (1 + df[b]).cumprod()
        feat[f'cumrel_{name}'] = cum_a / cum_b - 1

        # relative z-score
        roll_mean = spread.rolling(window).mean()
        roll_std  = spread.rolling(window).std()
        feat[f'zrel_{name}_{window}'] = (spread - roll_mean) / roll_std

        # rolling relative volatility (spread vol)
        feat[f'relvol_{name}_{window}'] = spread.rolling(window).std()

        # correlation between factors
        feat[f'corr_{a.lower()}_{b.lower()}_{window}'] = df[a].rolling(window).corr(df[b])

        # volatility ratio
        vol_a = df[a].rolling(window).std()
        vol_b = df[b].rolling(window).std()
        feat[f'volratio_{a.lower()}_{b.lower()}_{window}'] = vol_a / vol_b

    return feat


In [15]:
factor_relative_features = factor_relative_core_features(df, window=12)

In [16]:
# Save output into csv file:
factor_relative_features.to_csv('factor_relative_features.csv', index=True)